In [ ]:
from CausalSurv.data.datamodule_cv import ESMEOnlineDataModuleCV
from CausalSurv.data import ESMEProgressionOnlineDataModuleCV
import matplotlib.pyplot as plt


DATA_PATH = "/Users/malek/TheLAB/DynaSurv/data"
CONFIG_PATH = "/Users/malek/TheLAB/DynaSurv/configs/config_train.toml"
SPLIT_SEED = 2021779468
SUBTYPE = "HR+HER2-"
N_LINES = 4
BATCH_SIZE = 32
N_INTERVALS = 30
PROGESSION_COL = "Y_onset_to_progression"
DEATH_COL = "Y_onset_to_death"

In [ ]:
data_module_prog = ESMEProgressionOnlineDataModuleCV(
    data_dir=DATA_PATH,
    subtype=SUBTYPE,
    n_lines=N_LINES,
    n_intervals=N_INTERVALS,
    batch_size=BATCH_SIZE,
    split_seed=SPLIT_SEED,
    num_workers=0,
    final_training=True,
    bound_split="uniform"
)
data_module_prog.prepare_data()
data_module_prog.setup()
df = data_module_prog.raw_df

In [ ]:
death_bounds = data_module_prog.death_bounds
prog_bounds = data_module_prog.progression_bounds

fig, ax = plt.subplots(nrows=2, ncols=N_LINES, figsize=(3 * N_LINES, 6))
for line in range(1, N_LINES + 1):
    ax0 = ax[0, line - 1]
    ax1 = ax[1, line - 1]
    t_prog_line = df.loc[df["lineid"] == line, PROGESSION_COL]
    t_death_line = df.loc[df["lineid"] == line, DEATH_COL]

    ax0.hist(t_prog_line, bins=20)
    ax0.axvline(t_prog_line.median(), color="red", linestyle="--", label="Median")
    ax0.axvline(t_prog_line.quantile(0.99), color="green", linestyle="--", label="99th percentile")
    ax0.set_title(f"Line {line} - Time to progression")
    ax0.grid()

    ax1.hist(t_death_line, bins=20)
    ax1.axvline(t_death_line.median(), color="red", linestyle="--", label="Median")
    ax1.axvline(t_death_line.quantile(0.99), color="green", linestyle="--", label="99th percentile")
    ax1.set_title(f"Line {line} - Time to death")
    ax1.grid()

    for d_edge0, d_edge1 in zip(death_bounds[line-1][:-1:2], death_bounds[line-1][1::2]):
        ax0.fill_betweenx([0, 3000], d_edge0, d_edge1, color="red", alpha=0.1)

plt.tight_layout()




In [ ]:
# import numpy as np

# death_bounds = data_module_prog.death_bounds
# prog_bounds = data_module_prog.progression_bounds

# for line in range(1, N_LINES + 1):
#     t_death_line = df.loc[df["lineid"] == line, DEATH_COL]
#     t_prog_line = df.loc[df["lineid"] == line, PROGESSION_COL]
#     counts = np.histogram(t_death_line, bins=death_bounds[line-1])[0]
#     print(f"Line {line} - Death counts per bin: {counts}")
#     counts = np.histogram(t_prog_line, bins=prog_bounds[line-1])[0]
#     print(f"Line {line} - Progression counts per bin: {counts}")

In [ ]:
d = data_module_prog.get_data_dimensions()
print(*d.keys(), sep="\n")

In [ ]:
d["progression_time_bins"].dim()